# Criando um transformer do zero

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [9]:
MAX_SEQ_LEN = 30

In [ ]:
class PossitionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term  = torch.exp(torch.arrange(0,d_model, 2).float()*(-math.log(10000.0)/d_model) )
        
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)

    def forward(self, x):
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 512, num_heads=8):
        super(),__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num_heads'
        
        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        """
        
        """
    
    def scale_dot_product(self, Q, K, V):
        pass

class PositionFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        pass

    def forward(self, x):
        pass

class EncoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = PositionFeedForward(d_model, d_ff)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout1  = nn.Dropout(dropout)
        self.dropout2  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attention_score = self.self_attn(x, x, x, mask)

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([])
        
    def forward(self, x, mask=None):
        pass

class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        pass
    def forward(self, x, encoder_output, target_mask, encoder_mask):
        pass

In [20]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                input_vocab_size, target_vocab_size,
                max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size,  d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding     = PossitionalEmbedding(d_model, max_len)
        self.encoder           = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder           = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.ouput_layer       = nn.Linear(d_model, target_vocab_size)

    def forward(self, source, target):
        # masks do encoder 
        source_mask, target_mask = self.mask(source, target)
        # Embedding e encoding positional
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)
        
        # Decoder embedding & positional encoding
        target = decoder_embeding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        #Decoder
        output = self.decoder(target, encoder_ouput, target_mask, source_mask)

        return self.output_layer(output)


    def mask(self, source, target):
        source_mask = (source != 0 ).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0 ).unsqueeze(1).unsqueeze(2)

        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask

        return source_mask, target_mask